# Step 1 — Import Libraries

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# Step 2 — Load Dataset

In [2]:
# Load Dataset
california = fetch_california_housing(as_frame=True)
df = california.frame

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

print(df.head())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  


# Step 3 — Train-Test Split

In [3]:
# Step 3 — Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (16512, 8)
Testing shape: (4128, 8)


# Step 4 — Feature Scaling (IMPORTANT for Ridge & Lasso)

## Regularization is sensitive to feature scale.

In [4]:
# Step 4 — Feature Scaling (IMPORTANT for Ridge & Lasso)
# Regularization is sensitive to feature scale
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model 1 — Linear Regression (Baseline)

In [5]:
# Model 1 — Linear Regression (Baseline)
lin_model = LinearRegression()
lin_model.fit(X_train_scaled, y_train)

y_pred_lin = lin_model.predict(X_test_scaled)

mse_lin = mean_squared_error(y_test, y_pred_lin)
r2_lin = r2_score(y_test, y_pred_lin)

print("Linear Regression")
print(f"MSE: {mse_lin:.4f}")
print(f"R2:  {r2_lin:.4f}")

Linear Regression
MSE: 0.5559
R2:  0.5758


# Model 2 — Ridge Regression

In [6]:
# Model 2 — Ridge Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_model.predict(X_test_scaled)

mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print("\nRidge Regression")
print(f"MSE: {mse_ridge:.4f}")
print(f"R2:  {r2_ridge:.4f}")


Ridge Regression
MSE: 0.5559
R2:  0.5758


# Model 3 — Lasso Regression

In [7]:
# Model 3 — Lasso Regression
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train_scaled, y_train)

y_pred_lasso = lasso_model.predict(X_test_scaled)

mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

print("\nLasso Regression")
print(f"MSE: {mse_lasso:.4f}")
print(f"R2:  {r2_lasso:.4f}")


Lasso Regression
MSE: 0.6796
R2:  0.4814


# Step 5 — Compare Results in a Table

In [8]:
# Step 5 — Compare Results in a Table
results = pd.DataFrame({
    "Model": ["Linear", "Ridge", "Lasso"],
    "MSE": [mse_lin, mse_ridge, mse_lasso],
    "R2 Score": [r2_lin, r2_ridge, r2_lasso]
})

print(results)

    Model       MSE  R2 Score
0  Linear  0.555892  0.575788
1   Ridge  0.555855  0.575816
2   Lasso  0.679629  0.481361


# Step 6 — Compare Coefficients

In [9]:
# Step 6 — Compare Coefficients
coef_comparison = pd.DataFrame({
    "Feature": X.columns,
    "Linear": lin_model.coef_,
    "Ridge": ridge_model.coef_,
    "Lasso": lasso_model.coef_
})

print(coef_comparison)

      Feature    Linear     Ridge     Lasso
0      MedInc  0.854383  0.854327  0.710598
1    HouseAge  0.122546  0.122624  0.106453
2    AveRooms -0.294410 -0.294210 -0.000000
3   AveBedrms  0.339259  0.339008  0.000000
4  Population -0.002308 -0.002282 -0.000000
5    AveOccup -0.040829 -0.040833 -0.000000
6    Latitude -0.896929 -0.896168 -0.011469
7   Longitude -0.869842 -0.869071 -0.000000


## Model Comparison Analysis

- Linear Regression serves as the baseline model.
- Ridge Regression reduces coefficient magnitude but usually keeps all features.
- Lasso Regression may eliminate less important features by setting coefficients to zero.
- Compare MSE and R² to determine if regularization improved performance.

# step 8 - Test different alpha values

In [10]:
# Step 8 - Test different alpha values
for alpha in [0.01, 0.1, 1, 10]:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    print(f"Alpha={alpha}, R2={model.score(X_test_scaled, y_test):.4f}")

Alpha=0.01, R2=0.5758
Alpha=0.1, R2=0.5758
Alpha=1, R2=0.5758
Alpha=10, R2=0.5761
